## 井上绘图

In [ ]:
# ==============================================================================
# 按井绘制 HCT-QHead 与 HCT-PHead 预测结果
#
# 每口井输出一张一行三列图：
#   第 1 列：P-impedance（Ip）
#   第 2 列：S-impedance（Is）
#   第 3 列：Density（Dn）
#
# 输出示例：
#   ./analysis/W1.png
#   ./analysis/W2.png
#   ...
#   ./analysis/W20.png
#
# 说明：
#   - HCT-QHead 使用 q0.5 作为点预测；
#   - HCT-QHead 使用 q0.025 和 q0.975 绘制 95% 预测区间；
#   - HCT-PHead 绘制为蓝色虚线；
#   - 自动绘制 W1-W20，每张图以井名作为输出文件名；
#   - 三个子图标题分别为 P-impedance、S-impedance、Density；
#   - 每个子图分别显示对应单位；
#   - 不计算、不显示、不保存任何指标表格。
# ==============================================================================

import re
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, MultipleLocator

# ==============================================================================
# 0. Matplotlib 配置
# ==============================================================================
mpl.rcParams["font.sans-serif"] = "Noto Sans CJK JP"
mpl.rcParams["axes.unicode_minus"] = False


# ==============================================================================
# 1. 路径设置
# ==============================================================================
PROJECT_CANDIDATES = [
    Path("."),
    Path("AVO_CreatWell"),
]

for candidate in PROJECT_CANDIDATES:
    if (candidate / "HCT-QHead").exists() and (candidate / "HCT-PHead").exists():
        PROJECT_DIR = candidate
        break
else:
    raise FileNotFoundError("未找到同时包含 HCT-QHead/ 和 HCT-PHead/ 的项目目录。")


# HCT-QHead 和 HCT-PHead 的逐井预测结果目录
QHEAD_ROOT = PROJECT_DIR / "HCT-QHead" / "bywell"
PHEAD_ROOT = PROJECT_DIR / "HCT-PHead" / "bywell"


# 图片输出目录
ANALYSIS_DIR = Path("./analysis/well")
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)


# 每口井中三个子图的排列顺序
TARGET_ORDER = [
    "Ip",
    "Is",
    "Dn",
]


# ==============================================================================
# 2. 井范围设置
# ==============================================================================
# 数据集井号范围：
#   训练井：W1  - W15
#   验证井：W16 - W17
#   测试井：W18 - W20
#
# 程序会自动遍历全部井：
#   train -> W1  ~ W15
#   val   -> W16 ~ W17
#   test  -> W18 ~ W20
ROLE_WELL_RANGES = {
    "train": range(1, 16),
    "val": range(16, 18),
    "test": range(18, 21),
}

# 处理顺序
ROLE_ORDER = [
    "train",
    "val",
    "test",
]


def build_wells_by_role():
    """
    根据 ROLE_WELL_RANGES 自动生成每个数据集对应的全部井名。

    返回示例：
        {
            "train": ["W1", ..., "W15"],
            "val":   ["W16", "W17"],
            "test":  ["W18", "W19", "W20"],
        }
    """
    return {
        role: [f"W{i}" for i in ROLE_WELL_RANGES[role]]
        for role in ROLE_ORDER
    }


# ==============================================================================
# 3. 绘图配置
# ==============================================================================
# title：子图标题。
# xlabel：仅显示单位。
# xlim：对应参数的横坐标范围；设置为 None 时自动确定。
# x_major：横坐标主刻度间隔。
# formatter：横坐标刻度格式。
PLOT_CONFIG = {
    "Ip": {
        "title": "P-impedance",
        "xlabel": r"$\times 10^6$ kg/(m$^2$s)",
        "xlim": (4000.0, 12000.0),
        "x_major": 2000.0,
        "formatter": FuncFormatter(lambda value, position: f"{value / 1000:g}"),
    },
    "Is": {
        "title": "S-impedance",
        "xlabel": r"$\times 10^6$ kg/(m$^2$s)",
        "xlim": (2000.0, 8000.0),
        "x_major": 2000.0,
        "formatter": FuncFormatter(lambda value, position: f"{value / 1000:g}"),
    },
    "Dn": {
        "title": "Density",
        "xlabel": r"$\times 10^3$ kg/m$^3$",
        "xlim": (2.0, 3.0),
        "x_major": 0.25,
        "formatter": FuncFormatter(lambda value, position: f"{value:g}"),
    },
}


# 每个子图的宽度和高度
N_COLS = 3
# A5 横向尺寸：210 mm × 148 mm
FIG_W = 8.27
FIG_H = 5.83


# 绘图和保存 DPI
FIGURE_DPI = 300
SAVE_DPI = 300


# ==============================================================================
# 4. 工具函数
# ==============================================================================
def find_time_column(df):
    """
    自动识别时间列。

    识别以 Time 或 time 开头的列名。
    """
    for column in df.columns:
        if str(column).lower().startswith("time"):
            return column

    raise ValueError(f"未找到时间列。当前列名为：{list(df.columns)}")


def find_qhead_columns(df, target):
    """
    识别 HCT-QHead 输出中的：

        真实值列；
        预测区间下界列；
        P50 中位数预测列；
        预测区间上界列。
    """
    true_col = f"{target}_True"

    candidates = {
        "low": [
            f"{target}_Pred_q0.025",
            f"{target}_Pred_q0.0250",
            f"{target}_Pred_q2.5",
            f"{target}_Pred_P2.5",
            f"{target}_Pred_low",
        ],
        "mid": [
            f"{target}_Pred_q0.5",
            f"{target}_Pred_q0.50",
            f"{target}_Pred_q0.500",
            f"{target}_Pred_P50",
            f"{target}_Pred_median",
            f"{target}_Pred",
        ],
        "high": [
            f"{target}_Pred_q0.975",
            f"{target}_Pred_q0.9750",
            f"{target}_Pred_q97.5",
            f"{target}_Pred_P97.5",
            f"{target}_Pred_high",
        ],
    }

    if true_col not in df.columns:
        raise ValueError(
            f"HCT-QHead 文件中缺少真实值列：{true_col}。"
            f"当前列名为：{list(df.columns)}"
        )

    found = {}

    for key, candidate_columns in candidates.items():
        for column in candidate_columns:
            if column in df.columns:
                found[key] = column
                break

        if key not in found:
            raise ValueError(
                f"HCT-QHead 文件中缺少 {target} 的 {key} 预测列。"
                f"当前列名为：{list(df.columns)}"
            )

    return (
        true_col,
        found["low"],
        found["mid"],
        found["high"],
    )


def find_phead_columns(df, target):
    """
    识别 HCT-PHead 输出中的真实值列和点预测列。
    """
    true_col = f"{target}_True"

    pred_candidates = [
        f"{target}_Pred_PHead",
        f"{target}_Pred",
        f"{target}_PHead",
        "Pred_PHead",
        "Pred",
    ]

    if true_col not in df.columns:
        raise ValueError(
            f"HCT-PHead 文件中缺少真实值列：{true_col}。"
            f"当前列名为：{list(df.columns)}"
        )

    pred_col = None

    for column in pred_candidates:
        if column in df.columns:
            pred_col = column
            break

    if pred_col is None:
        raise ValueError(
            f"HCT-PHead 文件中缺少 {target} 的预测列。" f"当前列名为：{list(df.columns)}"
        )

    return true_col, pred_col


def read_selected_well_data(
    target,
    well_name,
    qhead_folder,
    phead_folder,
):
    """
    读取一口井某个参数对应的 HCT-QHead 和 HCT-PHead 预测结果。
    """
    qhead_file = qhead_folder / f"{well_name}.csv"
    phead_file = phead_folder / f"{well_name}.csv"

    if not qhead_file.exists():
        raise FileNotFoundError(f"HCT-QHead 结果文件不存在：{qhead_file}")

    if not phead_file.exists():
        raise FileNotFoundError(f"HCT-PHead 结果文件不存在：{phead_file}")

    df_qhead = pd.read_csv(qhead_file)
    df_phead = pd.read_csv(phead_file)

    time_col = find_time_column(df_qhead)

    (
        qhead_true_col,
        qhead_low_col,
        qhead_mid_col,
        qhead_high_col,
    ) = find_qhead_columns(
        df_qhead,
        target,
    )

    _, phead_pred_col = find_phead_columns(
        df_phead,
        target,
    )

    time = pd.to_numeric(
        df_qhead[time_col],
        errors="coerce",
    ).to_numpy(dtype=float)

    y_true = pd.to_numeric(
        df_qhead[qhead_true_col],
        errors="coerce",
    ).to_numpy(dtype=float)

    qhead_low = pd.to_numeric(
        df_qhead[qhead_low_col],
        errors="coerce",
    ).to_numpy(dtype=float)

    qhead_mid = pd.to_numeric(
        df_qhead[qhead_mid_col],
        errors="coerce",
    ).to_numpy(dtype=float)

    qhead_high = pd.to_numeric(
        df_qhead[qhead_high_col],
        errors="coerce",
    ).to_numpy(dtype=float)

    phead_pred = pd.to_numeric(
        df_phead[phead_pred_col],
        errors="coerce",
    ).to_numpy(dtype=float)

    # 如果两种模型结果长度略有差异，统一截断到最短长度。
    sample_count = min(
        len(time),
        len(y_true),
        len(qhead_low),
        len(qhead_mid),
        len(qhead_high),
        len(phead_pred),
    )

    if sample_count == 0:
        raise ValueError(f"{target} 的 {well_name} 没有可绘制数据。")

    return {
        "target": target,
        "well": well_name,
        "time": time[:sample_count],
        "y_true": y_true[:sample_count],
        "qhead_low": qhead_low[:sample_count],
        "qhead_mid": qhead_mid[:sample_count],
        "qhead_high": qhead_high[:sample_count],
        "phead_pred": phead_pred[:sample_count],
    }


def determine_xlim(
    item,
    plot_config,
    target,
):
    """
    确定当前井当前参数的横坐标范围。

    优先使用 PLOT_CONFIG 中设置的 xlim。
    当 xlim=None 时，根据当前井数据自动确定。
    """
    manual_xlim = plot_config.get("xlim")
    x_major = float(plot_config["x_major"])

    if manual_xlim is not None:
        if len(manual_xlim) != 2:
            raise ValueError(
                f'PLOT_CONFIG["{target}"]["xlim"] 必须是 '
                f"(x_min, x_max) 或 None，当前值为：{manual_xlim}"
            )

        x_min = float(manual_xlim[0])
        x_max = float(manual_xlim[1])

    else:
        finite_arrays = []

        for key in [
            "y_true",
            "qhead_low",
            "qhead_mid",
            "qhead_high",
            "phead_pred",
        ]:
            values = np.asarray(item[key], dtype=float)
            values = values[np.isfinite(values)]

            if values.size > 0:
                finite_arrays.append(values)

        if not finite_arrays:
            raise ValueError(
                f"{target} 的 {item['well']} 没有可用于确定横坐标范围的有限值。"
            )

        all_values = np.concatenate(finite_arrays)

        x_min = np.floor(float(np.nanmin(all_values)) / x_major) * x_major
        x_max = np.ceil(float(np.nanmax(all_values)) / x_major) * x_major

    if not np.isfinite(x_min) or not np.isfinite(x_max) or x_max <= x_min:
        raise ValueError(
            f"Target={target}，Well={item['well']} 的横坐标范围无效："
            f"x_min={x_min}, x_max={x_max}"
        )

    return x_min, x_max


def determine_common_ylim(well_plot_data):
    """
    根据同一口井的 Ip、Is、Dn 数据确定三个子图共用的时间范围。

    优先显示 2.10–2.60 s；
    如果三个参数不能共同覆盖该范围，则显示它们的公共完整时间范围。
    """
    time_mins = []
    time_maxs = []

    for target in TARGET_ORDER:
        time = np.asarray(well_plot_data[target]["time"], dtype=float)
        finite_time = time[np.isfinite(time)]

        if finite_time.size == 0:
            raise ValueError(
                f"{target} 的 {well_plot_data[target]['well']} 时间列没有有限值。"
            )

        time_mins.append(float(np.nanmin(finite_time)))
        time_maxs.append(float(np.nanmax(finite_time)))

    # 三个参数共同覆盖的时间范围
    common_time_min = max(time_mins)
    common_time_max = min(time_maxs)

    if common_time_min >= common_time_max:
        raise ValueError("Ip、Is、Dn 三个参数不存在共同的有效时间范围。")

    y_top = max(common_time_min, 2.10)
    y_bottom = min(common_time_max, 2.60)

    if y_top < y_bottom:
        return y_bottom, y_top

    return common_time_max, common_time_min


# ==============================================================================
# 5. 生成全部井列表
# ==============================================================================
wells_by_role = build_wells_by_role()

print(f"PROJECT_DIR : {PROJECT_DIR}")
print(f"QHEAD_ROOT    : {QHEAD_ROOT}")
print(f"PHEAD_ROOT    : {PHEAD_ROOT}")
print(f"ANALYSIS_DIR: {ANALYSIS_DIR}")

print()
print("将绘制以下井：")
for role in ROLE_ORDER:
    print(f"  {role:>5}: {', '.join(wells_by_role[role])}")


# ==============================================================================
# 6. 逐井读取并绘图
# ==============================================================================
for role in ROLE_ORDER:
    for well_name in wells_by_role[role]:
        # ----------------------------------------------------------------------
        # 6.1 读取当前井的 Ip、Is、Dn 数据
        # ----------------------------------------------------------------------
        well_plot_data = {}

        for target in TARGET_ORDER:
            qhead_folder = QHEAD_ROOT / target
            phead_folder = PHEAD_ROOT / target

            if not qhead_folder.exists():
                raise FileNotFoundError(f"HCT-QHead 参数目录不存在：{qhead_folder}")

            if not phead_folder.exists():
                raise FileNotFoundError(f"HCT-PHead 参数目录不存在：{phead_folder}")

            well_plot_data[target] = read_selected_well_data(
                target=target,
                well_name=well_name,
                qhead_folder=qhead_folder,
                phead_folder=phead_folder,
            )

        # --------------------------------------------------------------------------
        # 6.2 确定三个子图共用的时间范围
        # --------------------------------------------------------------------------
        y_bottom, y_top = determine_common_ylim(well_plot_data)

        # --------------------------------------------------------------------------
        # 6.3 创建当前井的一行三列图
        # --------------------------------------------------------------------------
        fig, axes = plt.subplots(
            1,
            N_COLS,
            figsize=(FIG_W, FIG_H),
            dpi=FIGURE_DPI,
            sharey=True,
        )

        axes = np.asarray(axes).reshape(-1)

        # # 整张图标题使用井名
        # fig.suptitle(
        #     well_name,
        #     fontsize=14,
        # )

        # --------------------------------------------------------------------------
        # 6.4 绘制 Ip、Is、Dn 三个子图
        # --------------------------------------------------------------------------
        for index, target in enumerate(TARGET_ORDER):
            ax = axes[index]
            item = well_plot_data[target]
            plot_config = PLOT_CONFIG[target]

            time = item["time"]
            y_true = item["y_true"]
            qhead_low = item["qhead_low"]
            qhead_mid = item["qhead_mid"]
            qhead_high = item["qhead_high"]
            phead_pred = item["phead_pred"]

            # 真实值：黑色实线
            ax.plot(
                y_true,
                time,
                color="black",
                lw=1.5,
                label="True",
            )

            # HCT-QHead P50：红色虚线
            ax.plot(
                qhead_mid,
                time,
                "--",
                color="red",
                lw=1.5,
                label="HCT-QHead P50",
            )

            # HCT-QHead 95% 预测区间
            ax.fill_betweenx(
                time,
                qhead_low,
                qhead_high,
                color="pink",
                alpha=0.99,
                label="HCT-QHead 95% PI",
            )

            # HCT-PHead：蓝色虚线
            ax.plot(
                phead_pred,
                time,
                "--",
                color="blue",
                lw=1.5,
                label="HCT-PHead",
            )

            # 子图标题：参数名称
            ax.set_title(
                plot_config["title"],
                fontsize=12,
            )

            # 每个子图均显示对应单位
            ax.set_xlabel(plot_config["xlabel"])

            # 仅第一列显示纵坐标标题
            if index == 0:
                ax.set_ylabel("Time(s)")
            else:
                ax.set_ylabel("")

            # 三个子图共用时间范围，时间向下增大
            ax.set_ylim(
                y_bottom,
                y_top,
            )

            # 当前参数横坐标范围
            x_min, x_max = determine_xlim(
                item=item,
                plot_config=plot_config,
                target=target,
            )

            ax.set_xlim(
                x_min,
                x_max,
            )

            # 横坐标主刻度
            ax.xaxis.set_major_locator(MultipleLocator(plot_config["x_major"]))

            # 横坐标数字格式
            ax.xaxis.set_major_formatter(plot_config["formatter"])

            # 时间轴主刻度间隔
            ax.yaxis.set_major_locator(MultipleLocator(0.05))

            # 网格样式
            ax.grid(
                True,
                ls="--",
                alpha=0.4,
            )

        # 原代码中图例未显示，因此不添加图例
        plt.tight_layout()

        # --------------------------------------------------------------------------
        # 6.5 以井名保存图片
        # --------------------------------------------------------------------------
        figure_path = ANALYSIS_DIR / f"{well_name}.png"

        fig.savefig(
            figure_path,
            dpi=SAVE_DPI,
        )

        print(f"已保存：{figure_path}")

        plt.show()

## 汇总表格

In [ ]:
# ==============================================================================
# 汇总三个参数、两个方法的指标表
#
# 输入：
#   HCT-PHead/bywell/metrics_summary_weighted_role_all_targets.csv
#   HCT-QHead/bywell/metrics_summary_weighted_role_all_targets.csv
#
# 输出：
#   analysis/well/HCT_all_methods_all_targets.csv
#
# 最终方法：
#   1. HCT-PHead
#   2. HCT-QHead
#
# 最终参数：
#   1. Dn
#   2. Ip
#   3. Is
#
# 最终数据集：
#   1. 训练
#   2. 测试（验证）
#   3. 测试（盲井）
#
# 说明：
#   - 两个方法分别读取各自按样本数加权的三参数指标表；
#   - 源表 Role 映射为数据集，相关系数统一使用 PCC；
#   - MARE 为平均绝对相对误差；
#   - RMSRE 为均方根相对误差；
#   - 最终 CSV 中连续重复的 Target 和方法会置空。
# ==============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# ==============================================================================
# 1. 路径设置
# ==============================================================================
CODE_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "Code",
    Path("/media/lug/Extreme SSD/文件/论文/AVO/QR-DNN-V5/Code"),
]

for candidate in CODE_CANDIDATES:
    if all((candidate / method / "bywell").is_dir()
           for method in ["HCT-PHead", "HCT-QHead"]):
        CODE_DIR = candidate.resolve()
        break
else:
    raise FileNotFoundError("未找到包含 HCT-PHead/bywell 和 HCT-QHead/bywell 的 Code 目录。")

METHOD_CSVS = {
    method: CODE_DIR / method / "bywell" / "metrics_summary_weighted_role_all_targets.csv"
    for method in ["HCT-PHead", "HCT-QHead"]
}
WELL_ANALYSIS_DIR = CODE_DIR / "analysis" / "well"
OUT_CSV = WELL_ANALYSIS_DIR / "HCT_all_methods_all_targets.csv"

for method, csv_path in METHOD_CSVS.items():
    print(f"{method:12}: {csv_path}")
print(f"OUT_CSV     : {OUT_CSV}")

# ==============================================================================
# 2. 表格配置
# ==============================================================================
TARGET_ORDER = [
    "Dn",
    "Ip",
    "Is",
]

METHOD_ORDER = [
    "HCT-PHead",
    "HCT-QHead",
]

ROLE_ORDER = [
    "训练",
    "测试（验证）",
    "测试（盲井）",
]

METRIC_COLUMNS = [
    "RMSRE",
    "MARE",
    "PCC",
    "PICP",
    "MPIW",
]


# ==============================================================================
# 3. 工具函数
# ==============================================================================
def normalize_text_column(series):
    """
    清理字符串列中的空白字符，并将空字符串转为 NaN。
    """
    out = series.astype("string").str.strip()

    out = out.replace(
        {
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA,
        }
    )

    return out


def normalize_role_name(value):
    """
    统一数据集名称，兼容可能出现的不同写法。
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    role_map = {
        "训练": "训练",
        "Train": "训练",
        "train": "训练",
        "Training": "训练",
        "training": "训练",
        "验证": "测试（验证）",
        "测试（验证）": "测试（验证）",
        "Validation": "测试（验证）",
        "validation": "测试（验证）",
        "Val": "测试（验证）",
        "val": "测试（验证）",
        "测试": "测试（盲井）",
        "盲井": "测试（盲井）",
        "测试（盲井）": "测试（盲井）",
        "Test": "测试（盲井）",
        "test": "测试（盲井）",
        "Blind test": "测试（盲井）",
        "blind test": "测试（盲井）",
    }

    return role_map.get(value, value)


def read_metric_table(csv_path, method):
    """读取该方法按数据集加权的三参数指标表，统一列名。"""
    df = pd.read_csv(csv_path)
    required_columns = ["Target", "Role", "MARE", "RMSRE", "PCC"]
    if method == "HCT-QHead":
        required_columns.extend(["PICP", "MPIW"])
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(
            f"{csv_path} 缺少必要列：{missing_columns}\n当前列：{list(df.columns)}"
        )

    df = df.rename(columns={"Role": "数据集"})
    df["方法"] = method
    df["Target"] = normalize_text_column(df["Target"])
    df["数据集"] = normalize_text_column(df["数据集"]).apply(normalize_role_name)

    # HCT-PHead 无预测区间，PICP 和 MPIW 留空。
    for col in METRIC_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan
        df[col] = pd.to_numeric(df[col], errors="raise")

    return df[["Target", "方法", "数据集", *METRIC_COLUMNS]].copy()


def check_expected_rows(df, table_name):
    """
    检查表中的参数、方法和数据集名称。
    """
    unknown_targets = sorted(set(df["Target"].dropna()) - set(TARGET_ORDER))

    unknown_roles = sorted(set(df["数据集"].dropna()) - set(ROLE_ORDER))

    if unknown_targets:
        print(f"Warning: {table_name} 存在未识别 Target：" f"{unknown_targets}")

    if unknown_roles:
        print(f"Warning: {table_name} 存在未识别数据集：" f"{unknown_roles}")


def validate_combined_table(df):
    """
    检查最终汇总表是否包含：
        3 个参数 × 2 个方法 × 3 个数据集 = 18 行。
    """
    expected_keys = {
        (target, method, role)
        for target in TARGET_ORDER
        for method in METHOD_ORDER
        for role in ROLE_ORDER
    }

    actual_keys = set(
        zip(
            df["Target"],
            df["方法"],
            df["数据集"],
        )
    )

    missing_keys = expected_keys - actual_keys
    extra_keys = actual_keys - expected_keys

    if missing_keys:
        missing_text = "\n".join(
            f"  {target}, {method}, {role}"
            for target, method, role in sorted(missing_keys)
        )

        raise ValueError("最终汇总表缺少以下组合：\n" f"{missing_text}")

    if extra_keys:
        extra_text = "\n".join(
            f"  {target}, {method}, {role}"
            for target, method, role in sorted(extra_keys)
        )

        print("Warning: 最终汇总表存在额外组合：\n" f"{extra_text}")

    duplicated = df.duplicated(
        subset=["Target", "方法", "数据集"],
        keep=False,
    )

    if duplicated.any():
        duplicated_rows = df.loc[
            duplicated,
            ["Target", "方法", "数据集"],
        ]

        raise ValueError(
            "最终汇总表存在重复组合：\n" f"{duplicated_rows.to_string(index=False)}"
        )


def format_metric_table(df):
    """
    将内部完整数据表格式化为最终 CSV 表格。

    处理内容：
        1. 指标保留 4 位小数；
        2. NaN 输出为空白；
        3. 连续重复的 Target 只显示一次；
        4. 连续重复的方法只显示一次；
        5. 将 Categorical 转为 object，避免写入空字符串时报错。
    """
    out = df.copy()

    # Categorical 不能直接写入新的空字符串类别。
    for col in ["Target", "方法", "数据集"]:
        out[col] = out[col].astype(object)

    # 指标保留 4 位小数。
    for col in METRIC_COLUMNS:
        out[col] = out[col].apply(lambda x: "" if pd.isna(x) else f"{float(x):.4f}")

    out = out[
        [
            "Target",
            "方法",
            "数据集",
            *METRIC_COLUMNS,
        ]
    ].copy()

    # Target 连续重复时置空。
    repeated_target = out["Target"].eq(out["Target"].shift())

    out.loc[
        repeated_target,
        "Target",
    ] = ""

    # 方法只在同一个 Target 内连续重复时置空。
    repeated_method = out["方法"].eq(out["方法"].shift()) & repeated_target

    out.loc[
        repeated_method,
        "方法",
    ] = ""

    return out


def display_left(df):
    """
    在 Jupyter Notebook 中将表格左对齐显示。
    """
    styler = df.style.set_properties(
        **{
            "text-align": "left",
        }
    ).set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "left"),
                ],
            },
        ]
    )

    try:
        styler = styler.hide(axis="index")
    except Exception:
        try:
            styler = styler.hide_index()
        except Exception:
            pass

    display(styler)


# ==============================================================================
# 4. 分别读取两个方法的三参数指标表
# ==============================================================================
method_tables = []
for method in METHOD_ORDER:
    table = read_metric_table(METHOD_CSVS[method], method)
    check_expected_rows(table, table_name=method)
    method_tables.append(table)


# ==============================================================================
# 5. 合并两个方法
# ==============================================================================
all_methods = pd.concat(method_tables, ignore_index=True)

# 只保留规定的参数、方法和数据集。
all_methods = all_methods[
    all_methods["Target"].isin(TARGET_ORDER)
    & all_methods["方法"].isin(METHOD_ORDER)
    & all_methods["数据集"].isin(ROLE_ORDER)
].copy()


# ==============================================================================
# 6. 检查完整性
# ==============================================================================
validate_combined_table(all_methods)


# ==============================================================================
# 7. 设置排序顺序
# ==============================================================================
all_methods["Target"] = pd.Categorical(
    all_methods["Target"],
    categories=TARGET_ORDER,
    ordered=True,
)

all_methods["方法"] = pd.Categorical(
    all_methods["方法"],
    categories=METHOD_ORDER,
    ordered=True,
)

all_methods["数据集"] = pd.Categorical(
    all_methods["数据集"],
    categories=ROLE_ORDER,
    ordered=True,
)

all_methods = all_methods.sort_values(
    [
        "Target",
        "方法",
        "数据集",
    ]
).reset_index(drop=True)


# ==============================================================================
# 8. 格式化并保存
# ==============================================================================
formatted = format_metric_table(all_methods)

WELL_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
formatted.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8-sig",
)

print()
print("=" * 80)
print(f"saved combined table: {OUT_CSV}")
print(f"rows: {len(formatted)}")

display_left(formatted)

## 导出 LaTeX 表格

In [ ]:
# ==============================================================================
# 将最终汇总 CSV 导出为论文使用的 LaTeX 表格
# ==============================================================================
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# ------------------------------------------------------------------------------
# 1. 文件与表格信息接口
# ------------------------------------------------------------------------------
# 与上方“汇总表格”使用同一输出目录；也支持单独运行本 cell。
LATEX_CODE_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "Code",
    Path("/media/lug/Extreme SSD/文件/论文/AVO/QR-DNN-V5/Code"),
]
for candidate in LATEX_CODE_CANDIDATES:
    if all((candidate / method / "bywell").is_dir()
           for method in ["HCT-PHead", "HCT-QHead"]):
        LATEX_DIR = candidate.resolve() / "analysis" / "well"
        break
else:
    raise FileNotFoundError("未找到 Code 目录，请先运行上方汇总表格 cell。")

LATEX_INPUT_CSV = LATEX_DIR / "HCT_all_methods_all_targets.csv"
LATEX_OUTPUT_TEX = LATEX_DIR / "HCT_all_methods_all_targets.tex"

TABLE_CAPTION = (
    "Performance comparison of different methods for "
    "elastic-parameter prediction."
)
TABLE_LABEL = "tab:model_compare_all_targets"

# CSV 名称 -> LaTeX 表格显示名称
TARGET_CONFIG = [
    ("Dn", "Density"),
    ("Ip", "P-impedance"),
    ("Is", "S-impedance"),
]

METHOD_CONFIG = [
    ("HCT-PHead", "HCT-PHead"),
    ("HCT-QHead", "HCT-QHead"),
]

ROLE_CONFIG = [
    ("训练", "Training"),
    ("测试（验证）", "Validation"),
    ("测试（盲井）", "Blind test"),
]

# LaTeX 表格中输出的指标及其顺序
METRIC_ORDER = [
    "RMSRE",
    "PCC",
    "PICP",
    "MPIW",
]


# ------------------------------------------------------------------------------
# 2. 工具函数
# ------------------------------------------------------------------------------
def read_latex_metrics(csv_path):
    """
    读取最终汇总表，并恢复为显示效果而置空的 Target 和方法列。
    """
    df = pd.read_csv(csv_path, dtype=str)

    required_columns = [
        "Target",
        "方法",
        "数据集",
        *METRIC_ORDER,
    ]

    missing_columns = [
        column for column in required_columns if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(f"{csv_path} 缺少必要列：{missing_columns}")

    for column in ["Target", "方法", "数据集"]:
        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

    df["Target"] = df["Target"].ffill()
    df["方法"] = df["方法"].ffill()

    return df


def format_latex_metric(value, metric):
    """
    格式化 LaTeX 表格中的指标数值。

    - 所有数值保留 4 位小数；
    - 缺失值输出为 --；
    - PICP 转换为百分数，即乘以 100。
    """
    missing_text = {
        "",
        "nan",
        "none",
        "<na>",
    }

    if pd.isna(value) or str(value).strip().lower() in missing_text:
        return "--"

    numeric_value = float(value)

    # PICP 从 [0, 1] 比例转换为百分数
    if metric == "PICP":
        numeric_value *= 100
        return f"{numeric_value:.2f}"

    return f"{numeric_value:.4f}"


def select_metric_row(df, target, method, role):
    """
    按 Target、方法和数据集唯一选取一行。
    """
    selected = df[
        df["Target"].eq(target)
        & df["方法"].eq(method)
        & df["数据集"].eq(role)
    ]

    if len(selected) != 1:
        raise ValueError(
            f"Target={target}, Method={method}, Dataset={role} "
            f"应有且仅有一行，当前为 {len(selected)} 行。"
        )

    return selected.iloc[0]


# ------------------------------------------------------------------------------
# 3. 读取数据并生成 LaTeX
# ------------------------------------------------------------------------------
metric_df = read_latex_metrics(LATEX_INPUT_CSV)

latex_lines = [
    r"\begin{table}[htp]",
    r"\centering",
    rf"\caption{{{TABLE_CAPTION}}}",
    rf"\label{{{TABLE_LABEL}}}",
    r"\renewcommand{\arraystretch}{1.10}",
    r"\setlength{\tabcolsep}{3pt}",
    r"\resizebox{\columnwidth}{!}{",
    r"    \begin{tabular}{cclcccc}",
    r"        \toprule",
    r"        Parameter & Method & Dataset & RMSRE & PCC & PICP(\%) & MPIW \\",
    r"        \midrule",
]

for target_index, (target_key, target_name) in enumerate(TARGET_CONFIG):
    for method_index, (method_key, method_name) in enumerate(METHOD_CONFIG):
        if method_index > 0:
            latex_lines.append(r"        \cmidrule(lr){2-7}")

        for role_index, (role_key, role_name) in enumerate(ROLE_CONFIG):
            row = select_metric_row(
                metric_df,
                target=target_key,
                method=method_key,
                role=role_key,
            )

            metric_values = " & ".join(
                format_latex_metric(
                    row[metric],
                    metric=metric,
                )
                for metric in METRIC_ORDER
            )

            if method_index == 0 and role_index == 0:
                target_cell = rf"\multirow{{6}}{{*}}{{{target_name}}}"
            else:
                target_cell = ""

            if role_index == 0:
                method_cell = rf"\multirow{{3}}{{*}}{{{method_name}}}"
            else:
                method_cell = ""

            latex_lines.append(
                f"        {target_cell} & {method_cell} & {role_name} "
                f"& {metric_values} " + r"\\"
            )

    if target_index < len(TARGET_CONFIG) - 1:
        latex_lines.extend(
            [
                "",
                r"        \midrule",
                "",
            ]
        )

latex_lines.extend(
    [
        r"        \bottomrule",
        r"    \end{tabular}",
        r"}",
        r"\end{table}",
    ]
)

latex_code = "\n".join(latex_lines) + "\n"


# ------------------------------------------------------------------------------
# 4. 保存并在 Notebook 中显示
# ------------------------------------------------------------------------------
LATEX_OUTPUT_TEX.parent.mkdir(
    parents=True,
    exist_ok=True,
)

LATEX_OUTPUT_TEX.write_text(
    latex_code,
    encoding="utf-8",
)

print(f"saved LaTeX table: {LATEX_OUTPUT_TEX}")
display(Markdown(f"```latex\n{latex_code}```"))

## 示例输入输出

In [ ]:
# W1：Seis15、Seis30、Seis45、Ip、Is、Dn 一行六列配对图
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FormatStrFormatter, MaxNLocator, MultipleLocator


# ------------------------------------------------------------------------------
# 1. 路径与输出配置
# ------------------------------------------------------------------------------
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "well" / "W1.csv").is_file():
            return candidate
    raise FileNotFoundError("未找到项目根目录 HCT-QHead_Synthesis_AVO。")


PROJECT_ROOT = find_project_root()
WELL_NAME = "W1"
WELL_CSV = PROJECT_ROOT / "well" / f"{WELL_NAME}.csv"
OUTPUT_DIR = PROJECT_ROOT / "Analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PNG = OUTPUT_DIR / f"well/{WELL_NAME}_Seismic_Elastic_1x6.png"
OUTPUT_PDF = OUTPUT_DIR / f"well/{WELL_NAME}_Seismic_Elastic_1x6.pdf"
SAVE_PDF = False
SHOW_FIGURE = True

# ------------------------------------------------------------------------------
# 2. 道配置与版式接口
# ------------------------------------------------------------------------------
TRACK_CONFIG = [
    {
        "column": "Seis15",
        "title": "Seismic 15°",
        "xlabel": r"$\times 10^3$",
        "scale": 1e3,
        "kind": "seismic",
    },
    {
        "column": "Seis30",
        "title": "Seismic 30°",
        "xlabel": r"$\times 10^3$",
        "scale": 1e3,
        "kind": "seismic",
    },
    {
        "column": "Seis45",
        "title": "Seismic 45°",
        "xlabel": r"$\times 10^3$",
        "scale": 1e3,
        "kind": "seismic",
    },
    {
        "column": "Ip",
        "title": "P-impedance",
        "xlabel": r"$\times 10^6$" + "\n" + r"kg/(m$^2$s)",
        "scale": 1e3,
        "kind": "elastic",
    },
    {
        "column": "Is",
        "title": "S-impedance",
        "xlabel": r"$\times 10^6$" + "\n" + r"kg/(m$^2$s)",
        "scale": 1e3,
        "kind": "elastic",
    },
    {
        "column": "Dn",
        "title": "Density",
        "xlabel": r"$\times 10^3$" + "\n" + r"kg/m$^3$",
        "scale": 1.0,
        "kind": "elastic",
    },
]

# A4 横向宽度 × A4 短边的 1/3。
FIGURE_SIZE_INCHES = (11.69/2, 8.27/1.5)
FONT_SIZE = 9  # 小五号
DISPLAY_DPI = 120
SAVE_DPI = 300
SUBPLOT_WSPACE = 0.08  # 六个子图之间的水平间距；数值越小，间隔越窄
SEISMIC_LINE_WIDTH = 0.7
ELASTIC_LINE_WIDTH = 0.9
X_MARGIN_RATIO = 0.05

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Noto Sans CJK JP", "DejaVu Sans"],
        "axes.unicode_minus": False,
        "font.size": FONT_SIZE,
        "axes.titlesize": FONT_SIZE,
        "axes.labelsize": FONT_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
    }
)

# ------------------------------------------------------------------------------
# 3. 读取 W1，并按真实弹性测井共同有效区间配对
# ------------------------------------------------------------------------------
well_data = pd.read_csv(WELL_CSV)
well_data.columns = [
    str(column).replace("\ufeff", "").strip() for column in well_data.columns
]

required_columns = ["time_s", *[item["column"] for item in TRACK_CONFIG]]
missing_columns = [column for column in required_columns if column not in well_data.columns]
if missing_columns:
    raise ValueError(f"{WELL_CSV} 缺少列：{missing_columns}")

for column in required_columns:
    well_data[column] = pd.to_numeric(well_data[column], errors="coerce")

time = well_data["time_s"].to_numpy(dtype=float)
valid_log = np.isfinite(time)
for column in ("Ip", "Is", "Dn"):
    values = well_data[column].to_numpy(dtype=float)
    valid_log &= np.isfinite(values) & (values != 0.0)

if not np.any(valid_log):
    raise ValueError(f"{WELL_NAME} 没有共同有效的 Ip、Is、Dn 数据。")

time = time[valid_log]
paired_data = {
    item["column"]: well_data[item["column"]].to_numpy(dtype=float)[valid_log]
    for item in TRACK_CONFIG
}

print(f"井：{WELL_NAME}")
print(f"配对时间范围：{time.min():.3f} ～ {time.max():.3f} s")
print(f"图片输出：{OUTPUT_PNG}")

# ------------------------------------------------------------------------------
# 4. 一行六列绘图
# ------------------------------------------------------------------------------
figure, axes = plt.subplots(
    1,
    6,
    figsize=FIGURE_SIZE_INCHES,
    dpi=DISPLAY_DPI,
    sharey=True,
)
figure.subplots_adjust(
    left=0.1,
    right=0.98,
    top=0.95,
    bottom=0.11,
    wspace=SUBPLOT_WSPACE,
)

for axis_index, (ax, config) in enumerate(zip(axes, TRACK_CONFIG)):
    values = paired_data[config["column"]] / config["scale"]
    valid = np.isfinite(time) & np.isfinite(values)
    track_time = time[valid]
    track_values = values[valid]

    if track_values.size == 0:
        raise ValueError(f"{WELL_NAME} 的 {config['column']} 没有有效数据。")

    if config["kind"] == "seismic":
        # 标准变面积显示：曲线为黑色，仅填充正振幅一侧。
        ax.plot(track_values, track_time, color="black", linewidth=SEISMIC_LINE_WIDTH)
        ax.fill_betweenx(
            track_time,
            0.0,
            track_values,
            where=track_values >= 0.0,
            facecolor="black",
            edgecolor="none",
            interpolate=True,
        )
        amplitude_limit = float(np.max(np.abs(track_values)))
        amplitude_limit *= 1.0 + X_MARGIN_RATIO
        ax.set_xlim(-amplitude_limit, amplitude_limit)
        ax.axvline(0.0, color="black", linewidth=0.4, zorder=0)
    else:
        ax.plot(track_values, track_time, color="black", linewidth=ELASTIC_LINE_WIDTH)
        value_min = float(np.min(track_values))
        value_max = float(np.max(track_values))
        margin = (value_max - value_min) * X_MARGIN_RATIO
        if margin == 0.0:
            margin = max(abs(value_min), 1.0) * X_MARGIN_RATIO
        ax.set_xlim(value_min - margin, value_max + margin)

    ax.set_title(config["title"], fontsize=FONT_SIZE, pad=4)
    ax.set_xlabel(config["xlabel"], fontsize=FONT_SIZE, labelpad=3)
    ax.set_ylim(float(np.max(time)), float(np.min(time)))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    if config["column"] in ("Ip", "Is"):
        # 数据绘图前已除以 1e3，因此图上间隔 2 对应原始值间隔 2000。
        ax.xaxis.set_major_locator(MultipleLocator(2.0))
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.0f"))
    elif config["column"] == "Dn":
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE, length=2.5, pad=2)

    if axis_index == 0:
        ax.set_ylabel("Time (s)", fontsize=FONT_SIZE, labelpad=4)
    else:
        ax.tick_params(axis="y", labelleft=False)

figure.savefig(OUTPUT_PNG, dpi=SAVE_DPI, bbox_inches=None, facecolor="white")
print(f"已保存 PNG：{OUTPUT_PNG}")

if SAVE_PDF:
    figure.savefig(OUTPUT_PDF, bbox_inches=None, facecolor="white")
    print(f"已保存 PDF：{OUTPUT_PDF}")

if SHOW_FIGURE:
    plt.show()
else:
    plt.close(figure)

## end